```
---
title: Co-factor molecules in ChEBI
tags: smallMolecules, SPARQL, ChEBI, cofactor
lang: en
version: 1.1
date: 2026-08-11
---
```

In [1]:
#import collections
#import importlib
import IPython
#import json
#import matplotlib.pyplot as plt
#import networkx as nx
#import numpy as np
#import os
import pandas
#import plotly.express as px
#import rdflib
#import rdflib.namespace
#import scipy.stats as stats
import sparqldataframe
from SPARQLWrapper import SPARQLWrapper, JSON
# import statsmodels.stats.multitest
#import sys

In [2]:
#specify no max value for the column width
pandas.set_option('display.max_colwidth', None)

In [3]:
#reactomeVersion = 96 # 2026-03-25
reactomeVersion = 97 # 2026-06-21
#chebiVersion = 252 # CHECKED BY QUERY BELOW
#endpointURL = "http://localhost:3030/reactome/query"
endpointURL = "http://localhost:3030/chebi/query"
#endpointURL = "http://localhost:3030/reactomeChEBI/query"
rdfFormat = "turtle"

In [4]:
taxon = "hsa" # Homo sapiens
#taxon = "sce" # Saccharomyces cerevisiae

taxonName = {}
taxonName["hsa"] = "Homo sapiens"
taxonName["sce"] = "Saccharomyces cerevisiae"

print("Taxon: {} ({})".format(taxon, taxonName[taxon]))

Taxon: hsa (Homo sapiens)


In [5]:
prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>

PREFIX CHEBI: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>

PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>

"""

In [6]:
query="""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?ontology ?versionIRI (REPLACE(REPLACE(STR(?versionIRI), 'http://purl.obolibrary.org/obo/chebi/', ''), '/chebi.owl', '') AS ?versionNumber)
WHERE {
  ?ontology rdf:type owl:Ontology .
  ?ontology owl:versionIRI ?versionIRI .
}
"""

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

chebiVersion = int(results["results"]["bindings"][0]["versionNumber"]["value"])
print("ChEBI version: {}".format(chebiVersion))

ChEBI version: 253


In [9]:
query = """
SELECT DISTINCT ?chebiID ?moleculeLabel ?roleDirectID ?roleDirectLabel ?roleID ?roleLabel 
WHERE {
  VALUES ?role { CHEBI:23357 } # cofactor

  ?molecule oboInOwl:id ?chebiID .
  ?molecule rdfs:label ?moleculeLabel .
  #?molecule rdfs:subClassOf ?restriction .
  ?molecule rdfs:subClassOf+ ?restriction .
  ?restriction rdf:type owl:Restriction .
  ?restriction owl:onProperty obo:RO_0000087 .
  ?restriction owl:someValuesFrom ?roleDirect .
  ?roleDirect oboInOwl:id ?roleDirectID .
  ?roleDirect rdfs:label ?roleDirectLabel .
  ?roleDirect rdfs:subClassOf* ?role .
  ?role oboInOwl:id ?roleID .
  ?role rdfs:label ?roleLabel .
}
#ORDER BY ?chebiID ?role
ORDER BY ?moleculeLabel ?roleLabel
"""


dfCofactor = sparqldataframe.query(endpointURL, prefixes+query)
print("Nb ChEBI cofactors: {}".format(len(set(dfCofactor['chebiID']))))
dfCofactor.to_csv("results/chebi_version{}_cofactor.tsv.gz".format(chebiVersion), sep="\t", header=True, index=False)

Nb ChEBI cofactors: 674


<div class="alert alert-block alert-info">
<b>Cofactors (ChEBI v253):</b> 
    <ul>
        <li>239 molecules have a role that is a descendant of Cofactor</li>
        <li>674 molecules are descendants of a molecule that has a role that is a descendant of Cofactor</li>
    </ul>
</div>

In [10]:
dfCofactor

,chebiID,moleculeLabel,roleDirectID,roleDirectLabel,roleID,roleLabel
0,CHEBI:60780,"(2R,2(1)S)-2(1),2(2)-dicarboxy-8-ethenyl-2,7,12,18-tetramethyl-2,2(1)-dihydrobenzo[b]porphyrin-13,17-dipropanoic acid",CHEBI:23357,cofactor,CHEBI:23357,cofactor
1,CHEBI:60775,"(2R,2(1)S)-8-ethenyl-2(1),2(2)-bis(methoxycarbonyl)-17-(3-methoxy-3-oxopropyl)-2,7,12,18-tetramethyl-2,2(1)-dihydrobenzo[b]porphyrin-13-propanoic acid",CHEBI:23357,cofactor,CHEBI:23357,cofactor
2,CHEBI:60778,"(2R,2(1)S)8-ethenyl-2(1),2(2)-bis(methoxycarbonyl)-13-(3-methoxy-3-oxopropyl)-2,7,12,18-tetramethyl-2,2(1)-dihydrobenzo[b]porphyrin-17-propanoic acid",CHEBI:23357,cofactor,CHEBI:23357,cofactor
3,CHEBI:60781,"(2S,2(1)R)-2(1),2(2)-dicarboxy-8-ethenyl-2,7,12,18-tetramethyl-2,2(1)-dihydrobenzo[b]porphyrin-13,17-dipropanoic acid",CHEBI:23357,cofactor,CHEBI:23357,cofactor
4,CHEBI:60779,"(2S,2(1)R)-8-ethenyl-2(1),2(2)-bis(methoxycarbonyl)-13-(3-methoxy-3-oxopropyl)-2,7,12,18-tetramethyl-2,2(1)-dihydrobenzo[b]porphyrin-17-propanoic acid",CHEBI:23357,cofactor,CHEBI:23357,cofactor
...,...,...,...,...,...,...
747,CHEBI:51216,zinc porphyrin,CHEBI:23357,cofactor,CHEBI:23357,cofactor
748,CHEBI:28783,zinc protoporphyrin,CHEBI:23357,cofactor,CHEBI:23357,cofactor
749,CHEBI:51220,zinc tetramesitylporphyrin,CHEBI:23357,cofactor,CHEBI:23357,cofactor
750,CHEBI:51221,zinc tetraphenylporphyrin,CHEBI:23357,cofactor,CHEBI:23357,cofactor


In [11]:
dfMoleculeLabel = pandas.read_csv("results/reactome_version{}_{}_chebiMolecules.tsv.gz".format(reactomeVersion, taxon), sep = "\t")

In [12]:
dfMoleculeLabel['isCofactor'] = False

In [13]:
setCofactors = set(dfCofactor['chebiID'])

for i, row in dfMoleculeLabel.iterrows():
    if row['chebiID'] in setCofactors:
        #print(row['chebiID'])
        dfMoleculeLabel.loc[dfMoleculeLabel['chebiID'] == row['chebiID'], ['isCofactor']] = True

In [14]:
dfMoleculeLabel[dfMoleculeLabel['isCofactor'] == True]

,chebiID,moleculeChebiLabel,isCofactor
4,CHEBI:18408,cobamamide,True
52,CHEBI:15361,pyruvate,True
55,CHEBI:33737,di-mu-sulfido-diiron(2+),True
58,CHEBI:57986,riboflavin(1-),True
119,CHEBI:16189,sulfate,True
...,...,...,...
1827,CHEBI:28938,ammonium,True
1947,CHEBI:16753,pantetheine,True
1986,CHEBI:55376,ferriheme b(1-),True
2024,CHEBI:21437,Mo-molybdopterin cofactor,True


In [15]:
print(len(dfMoleculeLabel))

2044


In [16]:
dfMoleculeLabel.to_csv("results/reactome_version{}_{}_MoleculeCofactor.tsv.gz".format(reactomeVersion, taxon), sep="\t", header=True, index=False)